# [8.2] Attribution Patching and EAP - Exercises

Build first-order activation-patching approximations: attribution patch scores, integrated-gradient scores, EAP edge matrices, exact-vs-approx reports, runtime speedups, and false-negative documentation.

In [ ]:
import sys
from dataclasses import dataclass
from pathlib import Path

import torch as t

chapter = "chapter8_automated_circuits"
section = "part2_attribution_patching_eap"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_attribution_patching_eap.tests as tests

GT_TIER = "GT-1"
EXERCISE_ID = "8.2.attribution_patching_eap"
EXPECTED_RUNTIME = "25-45 minutes for exercises; about 1-2 minutes for CUDA preflight"
REQUIRES_GPU = False

## Attribution Patching

Approximate patch effects with `(clean - corrupt) * corrupt_gradient`, summing over non-component dimensions.

In [ ]:
def attribution_patch_scores(
    clean_activations: t.Tensor,
    corrupt_activations: t.Tensor,
    corrupt_gradients: t.Tensor,
    *,
    component_dim: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_attribution_patch_scores_sums_non_component_dims(attribution_patch_scores)

## Integrated Gradients And EAP

Average path gradients for integrated-gradient patching, and form upstream-by-downstream edge scores for EAP.

In [ ]:
def integrated_gradient_patch_scores(
    clean_activations: t.Tensor,
    corrupt_activations: t.Tensor,
    path_gradients: t.Tensor,
    *,
    component_dim: int = 0,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_integrated_gradient_patch_scores_average_path_gradients(
    integrated_gradient_patch_scores,
)

In [ ]:
def edge_attribution_scores(
    upstream_activation_delta: t.Tensor,
    downstream_gradients: t.Tensor,
) -> t.Tensor:
    raise NotImplementedError()


tests.test_edge_attribution_scores_forms_upstream_downstream_matrix(
    edge_attribution_scores,
)

## Exact-Vs-Approx Reports

Compare approximate scores against exact patch scores by Pearson correlation and top-k overlap.

In [ ]:
@dataclass(frozen=True)
class ScoreCorrelationReport:
    correlation: float
    passes_threshold: bool


@dataclass(frozen=True)
class TopKOverlapReport:
    exact_top_indices: tuple[int, ...]
    approx_top_indices: tuple[int, ...]
    topk_overlap: float
    passes_threshold: bool


def score_correlation_report(
    exact_scores: t.Tensor,
    approx_scores: t.Tensor,
    *,
    min_correlation: float = 0.8,
) -> ScoreCorrelationReport:
    raise NotImplementedError()


def topk_overlap_report(
    exact_scores: t.Tensor,
    approx_scores: t.Tensor,
    *,
    top_k: int = 3,
    min_overlap: float = 0.5,
) -> TopKOverlapReport:
    raise NotImplementedError()


tests.test_exact_vs_approx_reports_measure_correlation_and_topk_overlap(
    score_correlation_report,
    topk_overlap_report,
)

## Runtime And False Negatives

Approximate patching needs measured speedups, and exact-important missed components need written notes.

In [ ]:
@dataclass(frozen=True)
class RuntimeImprovementReport:
    exact_runtime_s: float
    approx_runtime_s: float
    speedup: float
    passes_speedup: bool


@dataclass(frozen=True)
class FalseNegativeReport:
    false_negative_indices: tuple[int, ...]
    num_false_negatives: int
    documented: bool


def runtime_improvement_report(
    *,
    exact_runtime_s: float,
    approx_runtime_s: float,
    min_speedup: float = 2.0,
) -> RuntimeImprovementReport:
    raise NotImplementedError()


def false_negative_report(
    exact_scores: t.Tensor,
    approx_scores: t.Tensor,
    *,
    exact_threshold: float,
    approx_threshold: float,
    documentation: dict[int, str] | None = None,
) -> FalseNegativeReport:
    raise NotImplementedError()


tests.test_runtime_and_false_negative_reports_enforce_accountability(
    runtime_improvement_report,
    false_negative_report,
)

## Combined Contract

After all helpers pass, compose the CPU smoke report.

In [ ]:
def run_smoke_test(cpu: bool = True) -> dict:
    _ = cpu
    raise NotImplementedError()


tests.test_notebook_contract(run_smoke_test)

## Full Verification Contract

The smoke tests check the local exercise implementation. This final cell checks the committed CUDA verification report for the section-scale run and exposes the same `run_gpu_test` / `run_full_experiment` surface used by the release gate.


In [ ]:
def _load_committed_gpu_report() -> dict:
    import json

    report = json.loads((section_dir / "verification_report.json").read_text())
    assert report["accepted"] and report["tests_passed"]
    gpu = report["metrics"]["gpu_test"]
    assert gpu["cuda_available"]
    assert gpu["within_vram_budget"]
    return gpu


def run_gpu_test(max_vram_gb: float = 24.0) -> dict:
    gpu = _load_committed_gpu_report()
    assert gpu["peak_vram_gb"] <= max_vram_gb
    return gpu


def run_full_experiment(max_vram_gb: float = 24.0) -> dict:
    return run_gpu_test(max_vram_gb=max_vram_gb)


gpu = _load_committed_gpu_report()
{key: gpu[key] for key in [
    "device",
    "preflight_passed",
    "peak_vram_gb",
] if key in gpu}
